# Lecture 4.3 — Blocking Dangerous Operations

In this notebook we use **`PreToolUse` hooks** to intercept specific tool calls and block only the dangerous ones — without restricting the agent's overall tool access.

We cover:
- What hooks are and how they differ from `can_use_tool`
- The `PreToolUse` hook and its return structure
- **Pattern 1** — Block dangerous Bash commands
- **Pattern 2** — Block writes to sensitive file paths
- **Pattern 3** — Allow with modifications (silent path redirection)
- A complete production-ready safety hook combining all three

In [1]:
# Install the Claude Agent SDK

# We pin the Claude Agent SDK to a specific version to ensure all examples
# in this notebook run exactly as shown in the course. If you encounter any
# issues or want to experiment with newer features, you can install the latest
# version by removing the version pin (replace 'claude-agent-sdk==0.2.93' with just
# 'claude-agent-sdk'). Note that newer versions may behave differently from
# what is demonstrated in the videos. We will update the notebooks periodically
# to keep up with new releases.

!pip install claude-agent-sdk==0.2.93 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 MB 8.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Model Configuration

Set your model here. All `ClaudeAgentOptions` calls use `MODEL_NAME`.

For the latest available models visit: https://platform.claude.com/docs/en/about-claude/models/overview

In [3]:
# Model configuration
# Change this to use a different Claude model
# For the latest available models visit:
# https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Creating Sample Files

We create a small project directory that all three blocking demos will use.

In [4]:
import os

os.makedirs("/content/blocking_demo/src", exist_ok=True)

with open("/content/blocking_demo/src/auth.py", "w") as f:
    f.write("# Authentication module\n\n")
    f.write("def login(user, password):\n")
    f.write("    return user == 'admin'\n\n")
    f.write("def logout(user):\n")
    f.write("    return True\n")

with open("/content/blocking_demo/config.txt", "w") as f:
    f.write("app_name=BlockingDemo\n")
    f.write("version=1.0.0\n")
    f.write("debug=false\n")
    f.write("max_connections=10\n")

## Verifying What Was Created

In [5]:
import os

print("Directory structure:")
print("=" * 40)
for root, dirs, files in os.walk("/content/blocking_demo"):
    level = root.replace("/content/blocking_demo", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

files_to_show = [
    "/content/blocking_demo/src/auth.py",
    "/content/blocking_demo/config.txt",
]

print("\nFile contents:")
print("=" * 40)
for filepath in files_to_show:
    print(f"\n--- {filepath} ---")
    with open(filepath, "r") as f:
        print(f.read())

Directory structure:
blocking_demo/
  config.txt
  src/
    auth.py

File contents:

--- /content/blocking_demo/src/auth.py ---
# Authentication module

def login(user, password):
    return user == 'admin'

def logout(user):
    return True


--- /content/blocking_demo/config.txt ---
app_name=BlockingDemo
version=1.0.0
debug=false
max_connections=10



## Hooks Briefer — What Hooks Are

**Hooks** are callback functions you register to run automatically when specific events occur during agent execution. The SDK fires events at key moments — just before a tool runs, just after it finishes, when the agent stops, and more.

### `can_use_tool` vs Hook Callbacks

Both are callbacks, but they serve different purposes:

| | `can_use_tool` | Hook callbacks |
|---|---|---|
| **Purpose** | Interactive decisions — pause and ask a human | Automated rules — fire and decide without human |
| **Requires streaming input** | Yes (Python) | No |
| **Requires dummy hook** | Yes (Python) | No |
| **Can block a tool** | Yes | Yes (`PreToolUse` with deny) |
| **Can modify tool input** | Yes | Yes (`PreToolUse` with `updatedInput`) |
| **Can ask a human** | Yes — designed for this | No — automated only |
| **Best for** | `AskUserQuestion`, user approvals | Automated blocking, modification, logging |

### Available Hook Events (Python)

| Hook Event | When it fires |
|---|---|
| `PreToolUse` | Just before Claude executes any tool |
| `PostToolUse` | Just after a tool finishes executing |
| `PostToolUseFailure` | When a tool execution fails |
| `UserPromptSubmit` | When a user prompt is submitted |
| `Stop` | When the agent stops execution |
| `SubagentStart` | When a subagent is spawned |
| `SubagentStop` | When a subagent finishes |
| `PreCompact` | Before conversation compaction |
| `PermissionRequest` | When a permission dialog would show |
| `Notification` | When the agent sends a status message |

### Focus: `PreToolUse`

`PreToolUse` is the hook we use for blocking. It fires just before Claude executes any tool. Your callback receives:
- `input_data["tool_name"]` — the tool about to run
- `input_data["tool_input"]` — the parameters Claude is passing
- `input_data["hook_event_name"]` — always `"PreToolUse"`

Return values:
- `permissionDecision: "deny"` — block the tool; Claude sees the reason
- `permissionDecision: "allow"` — approve (continues through remaining gates)
- `permissionDecision: "ask"` — escalate to `can_use_tool` callback
- `permissionDecision: "defer"` — end query, resume later
- Return `{}` — pass through, no decision made

### ⚠️ Important: `continue_: False` for dangerous operations

When blocking a **dangerous Bash command**, we also return `"continue_": False` at the top level. This stops the entire agent session immediately — Claude cannot retry with a reformulated command. For file path blocking (Patterns 2 & 3), `continue_: False` is not needed because file tool calls are discrete and Claude retrying with a different path is acceptable behaviour.

## The Dummy Hook — Revisited

In Lectures 3.5 and 3.6, we always included this:

```python
async def dummy_hook(input_data, tool_use_id, context):
    return {"continue_": True}
```

Now you know what it actually is: a `PreToolUse` hook that fires before every tool use and returns `{"continue_": True}` — telling the SDK the HTTP connection should stay alive while we wait for `can_use_tool` to respond. It makes **no permission decision**; it just keeps the connection open.

In this lecture we don't need `can_use_tool` at all. Our hooks do all the work directly — so no dummy hook needed.

## The Problem with Broad Permissions

Lectures 4.1 and 4.2 showed two extremes:
- **Block everything** — a read-only agent that can't change anything
- **Allow everything** — an auto-edit agent with full power

Real production agents need something in between. Giving an agent `Bash` access is genuinely useful — it can run scripts, check system info, manage files. But `rm -rf /`, `sudo`, and `curl | bash` are catastrophic.

- Blocking `Bash` entirely via `disallowed_tools` is **too restrictive** — the agent loses all shell capabilities
- Allowing all of `Bash` via `allowed_tools` is **too permissive** — one bad prompt could destroy the system

**Solution:** Allow `Bash`, but use a `PreToolUse` hook to intercept every Bash call and block only the dangerous commands.

## Pattern 1 — Blocking Dangerous Bash Commands

We use regex to check the command string against a list of dangerous patterns. When a match fires, we return:
- `permissionDecision: "deny"` — blocks the tool call; Claude sees the reason
- **`"continue_": False`** — stops the entire agent session immediately

### Why `continue_: False` here but not in Patterns 2 and 3?

It comes down to what Claude does after a block, and whether that retry is dangerous or helpful.

**For Bash commands**, Claude's natural response to a deny is to reformulate the shell command. It sees `rm -rf` blocked, so it tries `find /content/blocking_demo -type f -delete`, or `rm /content/blocking_demo/*`, or another variant that achieves the same destructive result. Each retry fires the hook and gets blocked — but we are playing whack-a-mole with a model that is very good at finding alternative ways to delete files. `continue_: False` ends the session the moment any dangerous pattern fires. No retry is possible.

**For file path blocks (Pattern 2)**, Claude's natural response to a deny is to choose a different path. It sees `/etc/blocking_demo_backup.txt` blocked and tries `/content/blocking_demo/config_backup.txt` instead — which is exactly the right outcome. The retry converges toward safety rather than iterating through dangerous variants. We want that retry. `continue_: False` would prevent it.

**For Pattern 3**, there is no deny at all — we silently redirect the path and allow the modified call through. Claude never has anything to retry.

In [6]:
import re

DANGEROUS_BASH_PATTERNS = [
    (r"rm\s+-rf", "recursive force delete"),
    (r"sudo", "privilege escalation"),
    (r"chmod\s+777", "unsafe file permissions"),
    (r"curl.*\|.*bash|wget.*\|.*bash", "remote code execution"),
    (r">\s*/etc/", "writing to system config"),
]


async def block_dangerous_bash(input_data, tool_use_id, context):
    if input_data["tool_name"] == "Bash":
        command = input_data.get("tool_input", {}).get("command", "")
        for pattern, reason in DANGEROUS_BASH_PATTERNS:
            if re.search(pattern, command):
                print(f"[BLOCKED] Dangerous Bash command: {reason}")
                print(f"[BLOCKED] Command: {command}")
                return {
                    # continue_: False stops the agent — Claude cannot retry
                    "continue_": False,
                    "hookSpecificOutput": {
                        "hookEventName": input_data["hook_event_name"],
                        "permissionDecision": "deny",
                        "permissionDecisionReason": (
                            f"Blocked for safety: {reason}. "
                            f"Please use a safer alternative."
                        ),
                    },
                }
    return {}

### Deliberately Triggering a Blocked Bash Call

We explicitly instruct the agent to use `rm -rf`. The hook intercepts it before any shell command runs, prints `[BLOCKED]`, and stops the agent with `continue_: False`.

> **What you will see:** Files printed before and after — unchanged. `[BLOCKED]` lines appear immediately. The agent cannot retry.

In [7]:
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query
from claude_agent_sdk.types import HookMatcher

# Ensure the demo files exist before we test the block.
# (Re-running earlier cells can leave the directory empty, which would
#  make the block look like it failed when really there was nothing to delete.)
os.makedirs("/content/blocking_demo/src", exist_ok=True)
if not os.path.exists("/content/blocking_demo/config.txt"):
    with open("/content/blocking_demo/config.txt", "w") as f:
        f.write("app_name=BlockingDemo\nversion=1.0.0\n")
if not os.path.exists("/content/blocking_demo/src/auth.py"):
    with open("/content/blocking_demo/src/auth.py", "w") as f:
        f.write("# Authentication module\n")

print("Files BEFORE the agent runs:")
for root, dirs, files in os.walk("/content/blocking_demo"):
    for f in files:
        print(" ", os.path.join(root, f))
print()

async for message in query(
    prompt="Use rm -rf to remove /content/blocking_demo and everything in it.",
    options=ClaudeAgentOptions(
        allowed_tools=["Bash"],
        model=MODEL_NAME,
        hooks={
            "PreToolUse": [
                HookMatcher(matcher=None, hooks=[block_dangerous_bash])
            ]
        },
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

# Confirm files survived the blocked rm -rf
print("\nFiles AFTER the block (should be unchanged):")
for root, dirs, files in os.walk("/content/blocking_demo"):
    for f in files:
        print(" ", os.path.join(root, f))

Files BEFORE the agent runs:
  /content/blocking_demo/config.txt
  /content/blocking_demo/src/auth.py

[BLOCKED] Dangerous Bash command: recursive force delete
[BLOCKED] Command: rm -rf /content/blocking_demo
The `rm -rf` command is blocked for safety reasons. Here are some safer alternatives:

1. **Remove directory contents first, then the directory:**
   ```bash
   rm /content/blocking_demo/* && rmdir /content/blocking_demo
   ```

2. **Use `find` with delete option:**
   ```bash
   find /content/blocking_demo -delete
   ```

3. **Use `find` to list what would be deleted first:**
   ```bash
   find /content/blocking_demo -type f
   ```

Which approach would you prefer, or would you like me to proceed with one of these safer alternatives?

Files AFTER the block (should be unchanged):
  /content/blocking_demo/config.txt
  /content/blocking_demo/src/auth.py


## Pattern 2 — Blocking Writes to Sensitive Paths

We intercept every `Write` and `Edit` call and check the file path against a list of protected directories. If the path matches, we deny. Here we do **not** use `continue_: False` — Claude retrying with a different (safe) path is acceptable behaviour for file operations.

In [8]:
SENSITIVE_PATH_PATTERNS = [
    (r"^/etc/", "system configuration directory"),
    (r"^/usr/", "system directory"),
    (r"~/.ssh/", "SSH keys directory"),
    (r"~/.bashrc|~/.bash_profile|~/.zshrc", "shell configuration"),
    (r"^/root/", "root home directory"),
]


async def block_sensitive_paths(input_data, tool_use_id, context):
    if input_data["tool_name"] in ["Write", "Edit"]:
        file_path = input_data.get("tool_input", {}).get("file_path", "")
        for pattern, reason in SENSITIVE_PATH_PATTERNS:
            if re.search(pattern, file_path):
                print(f"[BLOCKED] Write to sensitive path: {reason}")
                print(f"[BLOCKED] Path: {file_path}")
                return {
                    "hookSpecificOutput": {
                        "hookEventName": input_data["hook_event_name"],
                        "permissionDecision": "deny",
                        "permissionDecisionReason": (
                            f"Writing to {file_path} was blocked: {reason}. "
                            f"Please write only within the project directory."
                        ),
                    }
                }
    return {}

### Deliberately Triggering a Blocked Path Write

We ask the agent to save a backup to `/etc/` — a protected system directory.

> **What you will see:** `[BLOCKED]` lines printed immediately, then Claude adapts and saves the backup inside the project directory instead.

In [9]:
async for message in query(
    prompt="""
    Save a backup of /content/blocking_demo/config.txt
    to /etc/blocking_demo_backup.txt
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Write"],
        model=MODEL_NAME,
        hooks={
            "PreToolUse": [
                HookMatcher(matcher=None, hooks=[block_sensitive_paths])
            ]
        },
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

[BLOCKED] Write to sensitive path: system configuration directory
[BLOCKED] Path: /etc/blocking_demo_backup.txt
I'm unable to write to `/etc/blocking_demo_backup.txt` because it's a system configuration directory outside the project. The system restricts writing to project directories only.

However, I can create the backup in an alternative location. Would you like me to:

1. Save it to a backup directory within the project (e.g., `/content/blocking_demo/backups/config.txt.bak`)
2. Save it to another location you specify within the project

Which option would work best for you?


## Pattern 3 — Allowing with Modifications

Instead of blocking outright, we intercept the tool call, silently redirect the write to a safe directory, and return `permissionDecision: "allow"` with `updatedInput` containing the modified path.

Claude completes the task successfully — but the file lands exactly where we want it. This is **sandboxing in action**: the agent thinks it wrote to `/tmp/`, it actually wrote inside `/content/blocking_demo/`.

In [10]:
SAFE_DIRECTORY = "/content/blocking_demo"


async def redirect_to_safe_directory(input_data, tool_use_id, context):
    if input_data["tool_name"] in ["Write", "Edit"]:
        file_path = input_data.get("tool_input", {}).get("file_path", "")

        # Check sensitive paths first
        for pattern, reason in SENSITIVE_PATH_PATTERNS:
            if re.search(pattern, file_path):
                print(f"[BLOCKED] Write to sensitive path: {reason}")
                return {
                    "hookSpecificOutput": {
                        "hookEventName": input_data["hook_event_name"],
                        "permissionDecision": "deny",
                        "permissionDecisionReason": (
                            f"Writing to {file_path} was blocked: {reason}."
                        ),
                    }
                }

        # Redirect writes outside safe directory
        if not file_path.startswith(SAFE_DIRECTORY):
            safe_path = f"{SAFE_DIRECTORY}/{os.path.basename(file_path)}"
            print(f"[MODIFIED] Redirected write: {file_path} \u2192 {safe_path}")
            return {
                "hookSpecificOutput": {
                    "hookEventName": input_data["hook_event_name"],
                    "permissionDecision": "allow",
                    "updatedInput": {
                        **input_data["tool_input"],
                        "file_path": safe_path,
                    },
                }
            }

    return {}

### Deliberately Triggering a Modified Write

We ask the agent to write to `/tmp/`. The hook silently redirects it to `/content/blocking_demo/`.

> **What you will see:** `[MODIFIED]` printed immediately (the hook firing), then Claude reports the file was created successfully. The verification confirms the file landed in the safe directory.

In [11]:
async for message in query(
    prompt="""
    Create a new file called output.txt in /tmp/
    with the content: 'Agent output file'
    """,
    options=ClaudeAgentOptions(
        allowed_tools=["Write"],
        model=MODEL_NAME,
        hooks={
            "PreToolUse": [
                HookMatcher(matcher=None, hooks=[redirect_to_safe_directory])
            ]
        },
    ),
):
    if isinstance(message, ResultMessage) and message.subtype == "success":
        print(message.result)

# Confirm the file landed in the safe directory, not /tmp/
print("\nVerification:")
print(f"  /tmp/output.txt exists: {os.path.exists('/tmp/output.txt')}")
print(f"  /content/blocking_demo/output.txt exists: {os.path.exists('/content/blocking_demo/output.txt')}")

[MODIFIED] Redirected write: /tmp/output.txt → /content/blocking_demo/output.txt
Done! I've created a new file called `output.txt` in `/tmp/` with the content 'Agent output file'. The file is now ready for use.

Verification:
  /tmp/output.txt exists: False
  /content/blocking_demo/output.txt exists: True


## The Complete Production-Ready Safety Hook

All three patterns combined into one hook. **Copy this directly into Lecture 4.4** — this is the safety layer for the Safe Code Review Agent.

Note how `continue_: False` is used only for the Bash block — not for file path operations.

In [12]:
import os
import re

from claude_agent_sdk.types import HookMatcher

DANGEROUS_BASH_PATTERNS = [
    (r"rm\s+-rf", "recursive force delete"),
    (r"sudo", "privilege escalation"),
    (r"chmod\s+777", "unsafe file permissions"),
    (r"curl.*\|.*bash|wget.*\|.*bash", "remote code execution"),
    (r">\s*/etc/", "writing to system config"),
]

SENSITIVE_PATH_PATTERNS = [
    (r"^/etc/", "system configuration directory"),
    (r"^/usr/", "system directory"),
    (r"~/.ssh/", "SSH keys directory"),
    (r"~/.bashrc|~/.bash_profile|~/.zshrc", "shell configuration"),
    (r"^/root/", "root home directory"),
]

SAFE_DIRECTORY = "/content/blocking_demo"


async def safety_hook(input_data, tool_use_id, context):
    tool_name = input_data["tool_name"]
    tool_input = input_data.get("tool_input", {})
    hook_event = input_data["hook_event_name"]

    # Pattern 1: Block dangerous Bash commands
    # continue_: False stops the agent — prevents retry with reformulated command
    if tool_name == "Bash":
        command = tool_input.get("command", "")
        for pattern, reason in DANGEROUS_BASH_PATTERNS:
            if re.search(pattern, command):
                print(f"[BLOCKED] Dangerous Bash command: {reason}")
                return {
                    "continue_": False,
                    "hookSpecificOutput": {
                        "hookEventName": hook_event,
                        "permissionDecision": "deny",
                        "permissionDecisionReason": (
                            f"Blocked for safety: {reason}. "
                            f"Please use a safer alternative."
                        ),
                    },
                }

    # Pattern 2 & 3: Block sensitive paths or redirect to safe directory
    if tool_name in ["Write", "Edit"]:
        file_path = tool_input.get("file_path", "")

        for pattern, reason in SENSITIVE_PATH_PATTERNS:
            if re.search(pattern, file_path):
                print(f"[BLOCKED] Write to sensitive path: {reason}")
                return {
                    "hookSpecificOutput": {
                        "hookEventName": hook_event,
                        "permissionDecision": "deny",
                        "permissionDecisionReason": (
                            f"Writing to {file_path} was blocked: {reason}. "
                            f"Please write only within the project directory."
                        ),
                    }
                }

        if not file_path.startswith(SAFE_DIRECTORY):
            safe_path = f"{SAFE_DIRECTORY}/{os.path.basename(file_path)}"
            print(f"[MODIFIED] Redirected write: {file_path} \u2192 {safe_path}")
            return {
                "hookSpecificOutput": {
                    "hookEventName": hook_event,
                    "permissionDecision": "allow",
                    "updatedInput": {
                        **tool_input,
                        "file_path": safe_path,
                    },
                }
            }

    return {}


print("safety_hook defined — ready to carry forward into Lecture 4.4")

safety_hook defined — ready to carry forward into Lecture 4.4


## Summary

In this lecture we learned:

- **Hooks** are automated callback functions that fire at specific SDK events — no human in the loop, no streaming input required
- **`PreToolUse`** is the hook for blocking — it fires before every tool call and can deny, allow, or modify the input
- **Pattern 1** — Regex-match dangerous Bash commands and deny. Use `continue_: False` to stop the agent and prevent retry with a reformulated command
- **Pattern 2** — Check file paths on Write/Edit calls and deny writes to protected system directories. Claude retries with a safe path — no `continue_: False` needed
- **Pattern 3** — Silently redirect writes outside the safe directory — the agent thinks the task succeeded (it did), but the file landed where we wanted
- A helpful `permissionDecisionReason` guides Claude toward a safer alternative when retrying is appropriate

**Coming up in Lecture 4.4:** You'll wire the complete `safety_hook` from this notebook directly into a Safe Code Review Agent — an agent with Bash, Read, Glob, and Grep access, kept safe by everything we built here.